# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIRˆ² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is available via a Croissant schema URL.


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and available records using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Available record sets: {[rs['@id'] for rs in getattr(metadata, 'recordSet', [])]}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s in the dataset.

We will list the record sets included in this dataset and display the fields (columns) defined for one as an example.


In [ ]:
import json

record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    raise ValueError("No record sets are defined in the dataset metadata.")

print("Record sets and fields available:")
record_set_fields = {}
for rs in record_sets:
    rs_id = rs['@id']
    print(f"- RecordSet @id: {rs_id} (name: {rs.get('name', '')})")
    fields = rs.get('field', [])
    fields = fields if isinstance(fields, list) else [fields]
    record_set_fields[rs_id] = []
    for field in fields:
        # The field entry may be a dict or a string @id
        field_id = field['@id'] if isinstance(field, dict) else field
        record_set_fields[rs_id].append(field_id)
        print(f"    - Field @id: {field_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Always use the correct record set and field `@id`s as above.

In [ ]:
# List all record set @ids
record_set_ids = list(record_set_fields.keys())
dataframes = {}

# Load each record set as a DataFrame
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records. Columns: {list(df.columns)}\n")
        else:
            print(f"No records were loaded for RecordSet {record_set_id}\n")
    except Exception as e:
        print(f"Error loading from RecordSet {record_set_id}: {e}")

# Select the primary (first) record set loaded for demonstration below:
primary_record_set_id = record_set_ids[0]
print(f"Working with primary record set: {primary_record_set_id}")

# Display a preview of the DataFrame
if primary_record_set_id in dataframes:
    print(dataframes[primary_record_set_id].head())
else:
    print("Primary DataFrame is empty.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filtering, normalization, categorization, outlier removal, or grouping by attributes. You can modify column and field `@id` references below as appropriate for your use case.

In [ ]:
# Choose a numeric field. We'll use the first numeric-like field found in DataFrame columns.
df = dataframes[primary_record_set_id]

numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is None:
    # Try to coerce columns to numeric if possible
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
        except:
            pass

if numeric_field:
    print(f"Using numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 1
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by another field if a categorical exists
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].nunique() < 10:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df)
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions and explore relationships between fields. Here, we'll plot the distribution of the selected numeric field and a boxplot by group if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric field to plot.")

## 6. Conclusion

This notebook demonstrated loading, overviewing, filtering, and visualizing the <b>Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells</b> dataset using the `mlcroissant` library. We explored available record sets and fields (always referenced by their `@id`s) and demonstrated basic filtering and normalization using a numeric field, concluding with simple visualizations.

**Next steps**: You can further dig into the biology, explore advanced statistics, or use the data in machine learning workflows. For deeper metadata, always review the record set and field descriptions available in the Croissant schema through the `metadata` object.